# Improved Training Pipeline with nn module

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

$$
x_{scaled} = \frac{x - \mu}{\sigma}
$$

In [ ]:
# Split in train test
X_train, X_test, Y_train, Y_test = train_test_split(
    df.iloc[:, :-1], 
    df.iloc[:,-1], 
    random_state=42, 
    test_size=0.2,
    stratify=df.iloc[:, -1] # Stratify based on target for better distribution
    )

# Scale the inputs
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # Learns the std and mean to scale
X_test = scaler.transform(X_test) # Uses the std and mean value from X_train to scale

In [ ]:
# Convert train, test data from numpy to torch type
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
Y_train = torch.from_numpy(Y_train.to_numpy(copy=True)).to(torch.float32).view(-1, 1)
Y_test = torch.from_numpy(Y_test.to_numpy(copy=True)).to(torch.float32).view(-1, 1)
    # Copying is the best practice

X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

## Define the model

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid() 
    
    def forward(self, features):
        out = self.linear(features) # z
        out = self.sigmoid(out) # y_pred 
        
        return out
    
    def loss_function(self, y_pred, y):
        return nn.functional.binary_cross_entropy(y_pred, y)

        

### Parameters

In [ ]:
learning_rate = 0.1
epochs = 50

## Training_pipeline

In [ ]:
model = SimpleNN(X_train.shape[1])

In [ ]:
for epoch in range(epochs):
    y_pred = model(X_train)
    
    # Loss calculate
    loss = model.loss_function(y_pred, Y_train)
    
    # Backward pass
    loss.backward()
    
    # Parameter update
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad
        
    # Zero gradients
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()
    
    # Print loss in each epoch
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")
    

## Evaluate

In [ ]:
with torch.no_grad():
    y_eval = model.forward(X_test)
    y_eval = (y_eval > 0.5).float()

y_eval_np = y_eval.cpu().numpy().ravel()    
y_test_np = Y_test.cpu().numpy().ravel()

accuracy = accuracy_score(y_test_np, y_eval_np)
precision = precision_score(y_test_np, y_eval_np)
recall = recall_score(y_test_np, y_eval_np)
f1 = f1_score(y_test_np, y_eval_np)

print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, "
      f"Recall: {recall:.4f}, F1 Score: {f1:.4f}")